In [1]:
import anndata as ad
import pandas as pd
import scanpy as sc
import os
import matplotlib.pyplot as plt
import numpy as np
from scipy import sparse


# Read in data 
## Assign clusters to meaningful names

In [4]:
#read in annotated h5ad
adatas = ad.read_h5ad('../../RPG/metabolomics_RPG/background_removed/all_organs_annotated_5_1_26.h5ad')
sal_glands = ad.read_h5ad('../../RPG/metabolomics_RPG/background_removed/salivary_glands_annotated_5_1_26.h5ad')


In [5]:
# change organ baser on cluster assignment
for cluster, organ in [('6', 'SMG'), ('10', 'SMG'), ('20', 'SMG'), ('13', 'SLG')]:
    adatas.obs.loc[adatas.obs['leiden'] == cluster, 'organ'] = organ
# remove spot with Organ == 'SLG' and x < 35000
adatas = adatas[~((adatas.obs['organ'] == 'SLG') & (adatas.obs['x'] < 35000)), :].copy()
# merge and rename clusters to be descriptive
#adatas.obs['leiden'] = adatas.obs['leiden_backup'].copy()
adatas.obs['leiden_backup'] = adatas.obs['leiden'].copy()

adatas.obs['leiden'] = adatas.obs['leiden'].astype(str)
adatas.obs.loc[adatas.obs['leiden'].isin(['32']), 'leiden'] = 'RPG secretory'
adatas.obs.loc[adatas.obs['leiden'].isin(['30']), 'leiden'] = 'RPG secretory'
adatas.obs.loc[adatas.obs['leiden'].isin(['43']), 'leiden'] = 'RPG secretory'
adatas.obs.loc[adatas.obs['leiden'].isin(['33']), 'leiden'] = 'RPG epithelial'
adatas.obs.loc[adatas.obs['leiden'].isin(['34']), 'leiden'] = 'RPG epithelial'
adatas.obs.loc[adatas.obs['leiden'].isin(['37']), 'leiden'] = 'RPG nerve'
adatas.obs.loc[adatas.obs['leiden'].isin(['42']), 'leiden'] = 'RPG nerve'
adatas.obs.loc[adatas.obs['leiden'].isin(['31']), 'leiden'] = 'RPG border'
adatas.obs.loc[adatas.obs['leiden'].isin(['38']), 'leiden'] = 'RPG muscle'
adatas.obs.loc[adatas.obs['leiden'].isin(['13']), 'leiden'] = 'SLG'
adatas.obs.loc[adatas.obs['leiden'].isin(['20']), 'leiden'] = 'SMG duct and vessels'
adatas.obs.loc[adatas.obs['leiden'].isin(['6']), 'leiden'] = 'SMG ducts'
adatas.obs.loc[adatas.obs['leiden'].isin(['5']), 'leiden'] = 'SMG acinar'
adatas.obs.loc[adatas.obs['leiden'].isin(['10']), 'leiden'] = 'SMG acinar'
adatas.obs.loc[adatas.obs['leiden'].isin(['18']), 'leiden'] = 'parotid border'
adatas.obs.loc[adatas.obs['leiden'].isin(['23']), 'leiden'] = 'parotid border'
adatas.obs.loc[adatas.obs['leiden'].isin(['4']), 'leiden'] = 'parotid dense'
adatas.obs.loc[adatas.obs['leiden'].isin(['19']), 'leiden'] = 'parotid lymph node'
adatas.obs.loc[adatas.obs['leiden'].isin(['16']), 'leiden'] = 'parotid connective and ducts'
adatas.obs.loc[adatas.obs['leiden'].isin(['9']), 'leiden'] = 'parotid undefined'
adatas.obs.loc[adatas.obs['leiden'].isin(['0']), 'leiden'] = 'parotid main'
adatas.obs.loc[adatas.obs['leiden'].isin(['2']), 'leiden'] = 'pancreas 1'
adatas.obs.loc[adatas.obs['leiden'].isin(['7']), 'leiden'] = 'pancreas 1'
adatas.obs.loc[adatas.obs['leiden'].isin(['8']), 'leiden'] = 'pancreas 2'
adatas.obs.loc[adatas.obs['leiden'].isin(['12']), 'leiden'] = 'pancreas 2'
adatas.obs.loc[adatas.obs['leiden'].isin(['1']), 'leiden'] = 'spleen 1'
adatas.obs.loc[adatas.obs['leiden'].isin(['11']), 'leiden'] = 'spleen 1'
adatas.obs.loc[adatas.obs['leiden'].isin(['3']), 'leiden'] = 'spleen 2'
adatas.obs.loc[adatas.obs['leiden'].isin(['14']), 'leiden'] = 'spleen 1'


adatas.obs['leiden'] = adatas.obs['leiden'].astype('category')
adatas.obs['leiden'].unique()

['SLG', 'SMG ducts', 'SMG acinar', 'SMG duct and vessels', 'pancreas 1', ..., 'parotid lymph node', 'parotid border', 'parotid dense', 'spleen 2', 'spleen 1']
Length: 19
Categories (19, object): ['RPG border', 'RPG epithelial', 'RPG muscle', 'RPG nerve', ..., 'parotid main', 'parotid undefined', 'spleen 1', 'spleen 2']

In [6]:
### merge and clusters to be more meaningfull
#create new column leiden_backup to preserve original leiden clusters
#sal_glands.obs['leiden'] = sal_glands.obs['leiden_backup'].copy()
#sal_glands.obs['leiden_backup'] = sal_glands.obs['leiden'].copy()

# change organ baser on cluster assignment
for cluster, organ in [('2', 'SMG'), ('3', 'SMG'), ('4', 'SMG'), ('6', 'SMG'), ('12', 'SMG'), ('8', 'SLG'), ('23', 'SLG')]:
    sal_glands.obs.loc[sal_glands.obs['leiden'] == cluster, 'organ'] = organ

# remove spot with Organ == 'SLG' and x < 35000
sal_glands = sal_glands[~((sal_glands.obs['organ'] == 'SLG') & (sal_glands.obs['x'] < 35000)), :].copy()
# merge and rename clusters to be descriptive

sal_glands.obs['leiden'] = sal_glands.obs['leiden'].astype(str)
sal_glands.obs.loc[sal_glands.obs['leiden'].isin(['10']), 'leiden'] = 'RPG secretory'
sal_glands.obs.loc[sal_glands.obs['leiden'].isin(['14']), 'leiden'] = 'RPG epithelial'
sal_glands.obs.loc[sal_glands.obs['leiden'].isin(['21']), 'leiden'] = 'RPG nerve'
sal_glands.obs.loc[sal_glands.obs['leiden'].isin(['16']), 'leiden'] = 'RPG border'
sal_glands.obs.loc[sal_glands.obs['leiden'].isin(['22']), 'leiden'] = 'RPG muscle'
sal_glands.obs.loc[sal_glands.obs['leiden'].isin(['23']), 'leiden'] = 'SLG duct'
sal_glands.obs.loc[sal_glands.obs['leiden'].isin(['8']), 'leiden'] = 'SLG'
sal_glands.obs.loc[sal_glands.obs['leiden'].isin(['12']), 'leiden'] = 'SMG duct and vessels'
sal_glands.obs.loc[sal_glands.obs['leiden'].isin(['2']), 'leiden'] = 'SMG acinar'
sal_glands.obs.loc[sal_glands.obs['leiden'].isin(['3']), 'leiden'] = 'SMG ducts'
sal_glands.obs.loc[sal_glands.obs['leiden'].isin(['4']), 'leiden'] = 'SMG acinar'
sal_glands.obs.loc[sal_glands.obs['leiden'].isin(['6']), 'leiden'] = 'SMG acinar'



sal_glands.obs.loc[sal_glands.obs['leiden'].isin(['9']), 'leiden'] = 'parotid border'
sal_glands.obs.loc[sal_glands.obs['leiden'].isin(['19']), 'leiden'] = 'parotid border'
sal_glands.obs.loc[sal_glands.obs['leiden'].isin(['1']), 'leiden'] = 'parotid dense'
sal_glands.obs.loc[sal_glands.obs['leiden'].isin(['11']), 'leiden'] = 'parotid lymph node'
sal_glands.obs.loc[sal_glands.obs['leiden'].isin(['15']), 'leiden'] = 'parotid connective and ducts'
sal_glands.obs.loc[sal_glands.obs['leiden'].isin(['20']), 'leiden'] = 'parotid border'
sal_glands.obs.loc[sal_glands.obs['leiden'].isin(['7']), 'leiden'] = 'parotid connective and ducts'
sal_glands.obs.loc[sal_glands.obs['leiden'].isin(['18']), 'leiden'] = 'parotid dense'
sal_glands.obs.loc[sal_glands.obs['leiden'].isin(['13']), 'leiden'] = 'parotid main'
sal_glands.obs.loc[sal_glands.obs['leiden'].isin(['0']), 'leiden'] = 'parotid main'
sal_glands.obs.loc[sal_glands.obs['leiden'].isin(['5']), 'leiden'] = 'parotid undefined'
sal_glands.obs.loc[sal_glands.obs['leiden'].isin(['17']), 'leiden'] = 'parotid dense'
sal_glands.obs.loc[sal_glands.obs['leiden'].isin(['24']), 'leiden'] = 'parotid undefined'


sal_glands.obs['leiden'] = sal_glands.obs['leiden'].astype('category')
sal_glands.obs['leiden'].unique()

['SLG', 'SMG ducts', 'SMG acinar', 'SMG duct and vessels', 'SLG duct', ..., 'parotid border', 'parotid connective and ducts', 'parotid main', 'parotid lymph node', 'parotid dense']
Length: 16
Categories (16, object): ['RPG border', 'RPG epithelial', 'RPG muscle', 'RPG nerve', ..., 'parotid dense', 'parotid lymph node', 'parotid main', 'parotid undefined']

In [ ]:
#filter out clusters corresponding to border and connective tissue
remove_border = adatas.obs['leiden'].isin(['RPG border', 'parotid border', 'parotid undefined', 'parotid lymph node', 'RPG muscle', 'RPG epithelial'])
adatas_sub = adatas[~remove_border, :].copy()

In [ ]:
#filter out clusters corresponding to border and connective tissue
remove_border = sal_glands.obs['leiden'].isin(['RPG border', 'parotid border', 'parotid undefined', 'parotid lymph node', 'RPG muscle', 'RPG epithelial'])
sal_glands_sub = sal_glands[~remove_border, :].copy()

# Fig 5 b and Fig S 14b
## Scatterplot of cleaned data (all border pixels removed), colored by assigned leiden clusters for 5b and by organ for S14b

In [7]:
fig5b = adatas.obs[['x', 'y', 'leiden', 'organ']].copy()
fig5b = fig5b[fig5b['leiden'] != 'RPG border']
os.makedirs('figure_source_csv', exist_ok=True)
fig5b.to_csv('figure_source_csv/fig5bs14b.csv', index=True)

# Fig 5 c and Fig S 14c
## UMAP of cleaned data (all borders apart from RPG border) removed, colored by assigned leiden clusters and by organ for S14c

In [8]:
#make csv with adatas.obsm['X_umap']
fig5c = pd.DataFrame(adatas.obsm['X_umap'], columns=['UMAP1', 'UMAP2'])
fig5c['leiden'] = adatas.obs['leiden'].values
fig5c['organ'] = adatas.obs['organ'].values
fig5c.to_csv('figure_source_csv/fig5cs14c.csv', index=True)

# Fig5D
## PCA of PC 1 vs 2 of all organs, RPG and Parotid are filtered to remove non-salivary clusters

In [ ]:
fig5d = pd.DataFrame(adatas_sub.obsm['X_pca'][:, :2], columns=['PCA1', 'PCA2'])
fig5d['organ'] = adatas_sub.obs['organ'].values
fig5d.to_csv('figure_source_csv/fig5d.csv', index=True)

# Fig 5 e
## PCA of PC 1 vs 2 of only salivary glands, filtered to remove non-salivary clusters

In [ ]:
fig5e = pd.DataFrame(sal_glands_sub.obsm['X_pca'][:, :2], columns=['PCA1', 'PCA2'])
fig5e['leiden'] = sal_glands_sub.obs['leiden'].values
fig5e.to_csv('figure_source_csv/fig5e.csv', index=True)

# Fig 5 f
## top 10 absolute PC1 and PC2 loadings of salivary glands PCA

In [139]:
# extract PC1 loadings
pc1_loadings_sal = sal_glands.varm['PCs'][:, 0]

# Create dataframe
df_sal = pd.DataFrame({
    "Metabolite": sal_glands.var_names,
    "Loading": pc1_loadings_sal
})

# Select top 10 by absolute contribution
df_top10_sal = (
    df_sal
    .assign(abs_loading=np.abs(df_sal["Loading"]))
    .sort_values("abs_loading", ascending=False)
    .head(10)
    .sort_values("Loading")
)
fig5f1 = df_top10_sal[['Metabolite', 'Loading']].copy()
fig5f1.to_csv('figure_source_csv/fig5f1.csv', index=True)

In [140]:
# Extract PC2 loadings
pc2_loadings_sal = sal_glands.varm['PCs'][:, 1]

# Create dataframe
df_sal_2 = pd.DataFrame({
    "Metabolite": sal_glands.var_names,
    "Loading": pc2_loadings_sal
})

# Select top 10 by absolute contribution
df_top10_sal_2 = (
    df_sal_2
    .assign(abs_loading=np.abs(df_sal_2["Loading"]))
    .sort_values("abs_loading", ascending=False)
    .head(10)
    .sort_values("Loading") 
)
fig5f2 = df_top10_sal_2[['Metabolite', 'Loading']].copy()
fig5f2.to_csv('figure_source_csv/fig5f2.csv', index=True)


# Fig S 14 g
## top10 absolute PC1 and PC2 loadings of all organs

In [9]:
# Extract PC1 loadings
pc1_loadings = adatas.varm['PCs'][:, 0]

# Create dataframe
df = pd.DataFrame({
    "Metabolite": adatas.var_names,
    "Loading": pc1_loadings
})

# Select top 10 by absolute contribution
df_top10 = (
    df
    .assign(abs_loading=np.abs(df["Loading"]))
    .sort_values("abs_loading", ascending=False)
    .head(10)
    .sort_values("Loading")  # for nicer plotting
)
figs14g1 = df_top10[['Metabolite', 'Loading']].copy()
figs14g1.to_csv('figure_source_csv/figs14g1.csv', index=True)

In [10]:
# Extract PC2 loadings
pc2_loadings = adatas.varm['PCs'][:, 1]

# Create dataframe
df_2 = pd.DataFrame({
    "Metabolite": adatas.var_names,
    "Loading": pc2_loadings
})

# Select top 10 by absolute contribution
df_top10_2 = (
    df_2
    .assign(abs_loading=np.abs(df_2["Loading"]))
    .sort_values("abs_loading", ascending=False)
    .head(10)
    .sort_values("Loading")  # for nicer plotting
)
figs14g2 = df_top10_2[['Metabolite', 'Loading']].copy()
figs14g2.to_csv('figure_source_csv/figs14g2.csv', index=True)

# Fig 5 g and Fig S 15 b
## 5g: Heatmap of metabolite expression in each pixel for top 10 differentially expressed metabolites per organ. Parotid and RPG filtered to remove non-salivary clusters
## S15b: Rank metabolite plot of top 20 features per organ, ranked by t-test score

In [13]:
# calculate ranked mz and dendrogram for adatas_sub
sc.tl.rank_genes_groups(adatas_sub, 'organ', method='t-test', key_added='rank_mz_organ_plot')
sc.tl.dendrogram(adatas_sub, groupby='organ', var_names=adatas_sub.var_names)

In [ ]:
#ranked mz as dataframe
#only keep top 10 ranked mz for each organ
ranked_mz = pd.DataFrame(adatas_sub.uns['rank_mz_organ_plot']['names'][:10])
ranked_mz

,RPG,SLG,SMG,pancreas,parotid,spleen
0,7-Methoxy-4-(trifluoromethyl)coumarin,N-Acetylneuraminic acid,Vinylacetylglycine,"Uridine 2',3'-cyclic phosphate",mz808.514,L-glyceraldehyde 3-phosphate
1,"5-(3',4',5'-Trihydroxyphenyl)-gamma-valerolact...","2-Deoxy-2,3-dehydro-n-acetyl-neuraminic acid",Aminoadipic acid,Cytidine 2'-phosphate,PI(16:0/16:0),"Glyceric acid 1,3-biphosphate"
2,unknown9 (mz316.9808),Andolast,"2-Deoxy-2,3-dehydro-n-acetyl-neuraminic acid",mz500.279,mz835.5356,unknown7 (mz286.9338)
3,unknown2-M+2 (mz146.8666),Pyruvic acid,mz425.0816,"Cytidine 2',3'-cyclic phosphate",PI 34:2,Adrenic acid
4,unknown2-M (mz144.8697),unknown4 (mz244.0826),Glycerol 3-phosphate,"3,5-Dinitrobenzoic acid",mz836.543,PE 40:5
5,gamma-Glutamylglutamic acid,"N-Ethyl-N-(6,6-dimethyl-2-hepten-4-ynyl)-3-((3...",Uridine,mz324.06,PI 38:6,mz556.3059
6,mz199.0014,mz315.0988,N-Acetylneuraminic acid,mz501.2818,PI 34:0,mz197.9401
7,PE O-34:2,mz314.0913,mz215.0325,mz528.2744,mz797.6564,unknown8 (mz286.9782)
8,Xanthine,mz227.0821,mz328.0591,mz651.0983,PI 36:1,mz347.0404
9,N-Acetylneuraminic acid,mz331.0937,p-coumaroyltriacetate,mz885.5513,PS(22:5/22:5),unknown11 (mz384.9953)


In [204]:
# extract underlying data for heatmap of ranked mz
group_order = adatas_sub.uns['dendrogram_organ']['categories_ordered']
names = pd.DataFrame(adatas_sub.uns['rank_mz_organ_plot']['names'])
cols = pd.MultiIndex.from_tuples(
    [(g, f) for g in group_order for f in names[g][:10]],
    names=['top_in_group', 'feature'])

X = adatas_sub.X[:, adatas_sub.var_names.get_indexer(cols.get_level_values('feature'))]
X = X.toarray() if sparse.issparse(X) else np.asarray(X)

out = pd.DataFrame(X, index=adatas_sub.obs_names, columns=cols)
out.insert(0, ('', 'organ'), pd.Categorical(adatas_sub.obs['organ'].astype(str),
                                            categories=group_order, ordered=True))
out.sort_values(('', 'organ'), kind='stable').to_csv('figure_source_csv/fig5g.csv')

In [19]:
df = sc.get.rank_genes_groups_df(adatas_sub, group=None, key='rank_mz_organ_plot')
top20 = df.groupby('group', observed=True).head(20).copy()
top20.insert(1, 'rank', top20.groupby('group', observed=True).cumcount() + 1)
top20.to_csv('figure_source_csv/figS15b.csv', index=True)

# Fig 5j
## correlation heatmap between organs, based on top 10 highest and lowest PC1 and PC2 loadings

In [ ]:
# calculate mean expression for each organ
mean_expr = pd.DataFrame(
    adatas_sub.X,
    index=adatas_sub.obs['organ'],
    columns=adatas_sub.var_names
).groupby(level=0).mean()

/var/folders/ft/9cnhb36s19ld2jkvqs5qst1r0000gn/T/ipykernel_51621/2769772535.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  mean_expr = pd.DataFrame(


In [ ]:
# extract top 10 positive and top 10 negative loadings for PC1
pc1_loadings = adatas_sub.varm['PCs'][:, 0]

# Create dataframe
pc1_loadings = pd.DataFrame({
    "Metabolite": adatas_sub.var_names,
    "Loading": pc1_loadings
})
# Select top 10 positive and top 10 negative by contribution
pc1_top10 = pd.concat([
    pc1_loadings.nlargest(10, 'Loading'),
    pc1_loadings.nsmallest(10, 'Loading')
]).sort_values("Loading")


#extract top 10 positive and top 10 negative loadings for PC2
pc2_loadings = adatas_sub.varm['PCs'][:, 1]

# Create dataframe
pc2_loadings = pd.DataFrame({
    "Metabolite": adatas_sub.var_names,
    "Loading": pc2_loadings
})
# Select top 10 positive and top 10 negative by contribution
pc2_top10 = pd.concat([
    pc2_loadings.nlargest(10, 'Loading'),
    pc2_loadings.nsmallest(10, 'Loading')
]).sort_values("Loading")

In [ ]:
#filter mean_expr to only include metabolites in pc1_top10 or pc2_top10
top_metabolites = pd.concat([pc1_top10['Metabolite'], pc2_top10['Metabolite']]).unique()
mean_expr_top = mean_expr.loc[:, mean_expr.columns.isin(top_metabolites)]
corr_top = mean_expr_top.T.corr(method='spearman')

In [219]:
corr_top
#reorder the correlation matrix based on hierarchical clustering
corr_top_reordered = corr_top.loc[['pancreas', 'spleen', 'parotid', 'RPG', 'SLG', 'SMG'], ['pancreas', 'spleen', 'parotid', 'RPG', 'SLG', 'SMG']]
corr_top_reordered.to_csv('figure_source_csv/fig5j.csv', index=True)


# Fig5 k
# venn diagram of shared expressed metabolites between salivary glands

In [227]:
expressed_mets = pd.read_csv('metabolites_by_organ.csv')
expressed_mets[['RPG','SLG','SMG', 'parotid','id']]
expressed_mets.to_csv('figure_source_csv/fig5k.csv', index=False)